In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../config/samples.tsv", sep="\t")
df.head(3)

In [167]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def volcano_plot(
    data: pd.DataFrame, title: str, genes_to_label_df: pd.DataFrame = None, x_label=8, min_gap=0.6, fontsize=10
):
    # Assign colors
    def determine_color(row):
        if row["padj"] < 0.05 and abs(row["log2FoldChange"]) > 1:
            return "#ff6666"
        elif row["padj"] > 0.05 and abs(row["log2FoldChange"]) > 1:
            return "#39e600"
        elif row["padj"] < 0.05 and abs(row["log2FoldChange"]) < 1:
            return "#3385ff"
        else:
            return "#b3b3b3"

    data = data.copy()
    data["col"] = data.apply(determine_color, axis=1)

    # Labels (fallback GeneID if no GeneName)
    data["Gene_name"] = data["GeneName"].fillna(data["GeneID"])
    data.loc[data["Gene_name"] == "", "Gene_name"] = data["GeneID"]

    # Precompute -log10(p)
    data["-log10padj"] = -np.log10(data["padj"])

    fig = go.Figure()

    # Background/main scatter
    fig.add_trace(
        go.Scatter(
            x=data["log2FoldChange"],
            y=data["-log10padj"],
            mode="markers",
            marker=dict(color=data["col"], opacity=0.5, size=6),
            text=data["Gene_name"],
            hoverinfo="text+x+y",
        )
    )

    # Prevent overlapping labels
    def place_labels(labels, side="right", x_label=x_label, min_gap=min_gap, fontsize=fontsize):
        labels = labels.sort_values("-log10padj").reset_index(drop=True)
        y_positions = []
        last_y = -np.inf
        for y in labels["-log10padj"]:
            if y < last_y + min_gap:
                y = last_y + min_gap
            y_positions.append(y)
            last_y = y
        labels["y_label"] = y_positions

        for _, row in labels.iterrows():
            # Annotation text
            fig.add_annotation(
                x=x_label if side == "right" else -x_label,
                y=row["y_label"],
                text=row["Gene_name"],
                showarrow=False,
                xanchor="left" if side == "right" else "right",
                font=dict(color="black", size=fontsize),
            )

            # Leader line
            fig.add_shape(
                type="line",
                x0=row["log2FoldChange"],
                y0=row["-log10padj"],
                x1=x_label if side == "right" else -x_label,
                y1=row["y_label"],
                line=dict(color="grey", width=2, dash="dot"),
                layer="below",
            )

        # Also highlight labelled points: add another scatter layer
        fig.add_trace(
            go.Scatter(
                x=labels["log2FoldChange"],
                y=labels["-log10padj"],
                mode="markers",
                marker=dict(
                    color=labels["col"],
                    size=10,
                    line=dict(color="black", width=0.8),  # outline
                ),
                text=labels["Gene_name"],
                hoverinfo="text+x+y",
                showlegend=False,
            )
        )

    # Add labels if provided
    if genes_to_label_df is not None and not genes_to_label_df.empty:
        labels = data.loc[data["GeneID"].isin(genes_to_label_df["GeneID"]), :].copy()

        up = labels[labels["log2FoldChange"] >= 0]
        down = labels[labels["log2FoldChange"] < 0]

        if not up.empty:
            place_labels(up, side="right", x_label=8)
        if not down.empty:
            place_labels(down, side="left", x_label=8)

    # Formatting
    fig.update_layout(
        template="simple_white",
        title=title,
        xaxis_title="log2 Fold Change",
        yaxis_title="-log10(P)",
        showlegend=False,
        width=600,
        height=600,
    )

    fig.update_xaxes(range=[-10, 10])
    fig.update_yaxes(range=[0, 20])

    # Threshold lines
    fig.add_hline(y=-np.log10(0.05), line_dash="dash", line_color="grey")
    fig.add_vline(x=np.log2(2), line_dash="dash", line_color="grey")
    fig.add_vline(x=np.log2(0.5), line_dash="dash", line_color="grey")

    fig.update_layout(
        font=dict(
            family="DiverdaSansW01-Regular",  # must match system font name
            size=14,
            color="black"
        )
    )

    return fig

def find_genes(df, string, padj_threshold=0.05, lfc_threshold=1):
    df=  df.query(f"padj < {padj_threshold} & abs(log2FoldChange) > {lfc_threshold}").fillna('  ')
    return df.query(f"GeneDescription.str.contains('{string}')")

def label_genes(df):

    esterase = find_genes(df, 'esterase')
    P450 = find_genes(df, 'P450')
    gste = find_genes(df, 'glutathione')
    abc = find_genes(df, 'ABC')
    ace1 = find_genes(df, 'Acetylcholinesterase')
    OR = find_genes(df, 'odorant')

    return pd.concat([esterase, P450, gste, abc, ace1, OR])

In [ ]:
comps = ['gambiaeCont_gambiaePM','coluzziiCont_coluzziiPM']#, 'Kisumu_gambiaeCont', 'Ngousso_coluzziiCont']

In [ ]:
gdiff = {}
for comp in  comps:
    gdiff[comp] = pd.read_csv(f'../old_results/genediff/{comp}.csv')

In [ ]:
def find_genes(df, string, padj_threshold=0.05, lfc_threshold=1):
    df=  df.query(f"padj < {padj_threshold} & abs(log2FoldChange) > {lfc_threshold}").fillna('  ')
    return df.query(f"GeneDescription.str.contains('{string}')")


esterase = find_genes(gdiff['gambiaeCont_gambiaePM'], 'esterase')
P450 = find_genes(gdiff['gambiaeCont_gambiaePM'], 'P450')
gste = find_genes(gdiff['gambiaeCont_gambiaePM'], 'glutathione')
abc = find_genes(gdiff['gambiaeCont_gambiaePM'], 'ABC')
ace1 = find_genes(gdiff['gambiaeCont_gambiaePM'], 'Acetylcholinesterase')
OR = find_genes(gdiff['gambiaeCont_gambiaePM'], 'odorant')

In [171]:
for comp in  comps:
    genes_to_label_df = label_genes(gdiff[comp])
    fig = volcano_plot(gdiff[comp], title=comp, genes_to_label_df=genes_to_label_df, x_label=0, min_gap=0.65, fontsize=12)
    # fig.show()
    
    fig.write_image(f'{comp}-volcano.svg', scale=1)